In [0]:
# ============================================================
# 04_document_metadata
# ============================================================
#
# Purpose:
#   Create document-level metadata for the five World Bank
#   Global Economic Prospects reports and attach that metadata
#   to the deterministic parent/child chunk baseline.
#
# Inputs:
#   worldbank_ai.silver.gep_parent_chunks
#   worldbank_ai.silver.gep_child_chunks
#
# Outputs:
#   worldbank_ai.silver.gep_document_metadata
#   worldbank_ai.silver.gep_parent_chunks_enriched
#   worldbank_ai.silver.gep_child_chunks_enriched
#
# Important:
#   - No embeddings are created here.
#   - No LLM calls are made here.
#   - No source text is modified.
#   - Original chunk tables remain unchanged.
# ============================================================

CATALOG = "worldbank_ai"
SCHEMA = "silver"

PARENT_SOURCE_TABLE = (
    f"{CATALOG}.{SCHEMA}.gep_parent_chunks"
)

CHILD_SOURCE_TABLE = (
    f"{CATALOG}.{SCHEMA}.gep_child_chunks"
)

DOCUMENT_METADATA_TABLE = (
    f"{CATALOG}.{SCHEMA}.gep_document_metadata"
)

PARENT_ENRICHED_TABLE = (
    f"{CATALOG}.{SCHEMA}.gep_parent_chunks_enriched"
)

CHILD_ENRICHED_TABLE = (
    f"{CATALOG}.{SCHEMA}.gep_child_chunks_enriched"
)

REPORT_SERIES = "Global Economic Prospects"
REPORT_MONTH = 1
PUBLISHER = "World Bank"

print("Configuration loaded.")
print(f"Parent source:  {PARENT_SOURCE_TABLE}")
print(f"Child source:   {CHILD_SOURCE_TABLE}")
print(f"Metadata table: {DOCUMENT_METADATA_TABLE}")

In [0]:
# ============================================================
# Imports
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T

print("Imports loaded.")

In [0]:
# ============================================================
# Load deterministic chunk baseline
# ============================================================

parent_df = spark.table(PARENT_SOURCE_TABLE)
child_df = spark.table(CHILD_SOURCE_TABLE)

parent_count = parent_df.count()
child_count = child_df.count()

print(f"Parent chunks: {parent_count:,}")
print(f"Child chunks:  {child_count:,}")

print("\nParent columns:")
print(parent_df.columns)

print("\nChild columns:")
print(child_df.columns)

In [0]:
# ============================================================
# Validate expected document corpus
# ============================================================

parent_documents = (
    parent_df
    .select(
        "document_id",
        "report_year"
    )
    .distinct()
    .orderBy("report_year")
)

child_documents = (
    child_df
    .select(
        "document_id",
        "report_year"
    )
    .distinct()
    .orderBy("report_year")
)

print("Documents represented in parent chunks:")
display(parent_documents)

print("Documents represented in child chunks:")
display(child_documents)


# ------------------------------------------------------------
# Validate expected five-report corpus.
# ------------------------------------------------------------

expected_years = {
    2022,
    2023,
    2024,
    2025,
    2026
}

actual_parent_years = {
    row["report_year"]
    for row in parent_documents.collect()
}

actual_child_years = {
    row["report_year"]
    for row in child_documents.collect()
}

if actual_parent_years != expected_years:
    raise RuntimeError(
        "Unexpected report years in parent chunks. "
        f"Expected {expected_years}, "
        f"found {actual_parent_years}"
    )

if actual_child_years != expected_years:
    raise RuntimeError(
        "Unexpected report years in child chunks. "
        f"Expected {expected_years}, "
        f"found {actual_child_years}"
    )

print("Corpus validation passed.")

In [0]:
# ============================================================
# Inspect canonical document identifiers
# ============================================================

document_ids_df = (
    parent_df
    .select(
        "document_id",
        "report_year"
    )
    .distinct()
    .orderBy("report_year")
)

display(document_ids_df)

In [0]:
# ============================================================
# Build canonical document metadata
# ============================================================
#
# Important:
#   We derive the document IDs from the actual chunk table.
#
#   Publication dates and data-cutoff dates are intentionally
#   left null until independently verified from source material.
# ============================================================

document_metadata_schema = T.StructType([
    T.StructField("report_year", T.IntegerType(), False),
    T.StructField("report_series", T.StringType(), False),
    T.StructField("report_month", T.IntegerType(), False),
    T.StructField("edition_status", T.StringType(), False),
    T.StructField("publisher", T.StringType(), False),
    T.StructField("publication_date", T.DateType(), True),
    T.StructField("data_cutoff_date", T.DateType(), True),
])

metadata_rows = [
    (
        2022,
        REPORT_SERIES,
        REPORT_MONTH,
        "final",
        PUBLISHER,
        None,
        None,
    ),
    (
        2023,
        REPORT_SERIES,
        REPORT_MONTH,
        "final",
        PUBLISHER,
        None,
        None,
    ),
    (
        2024,
        REPORT_SERIES,
        REPORT_MONTH,
        "final",
        PUBLISHER,
        None,
        None,
    ),
    (
        2025,
        REPORT_SERIES,
        REPORT_MONTH,
        "final",
        PUBLISHER,
        None,
        None,
    ),
    (
        2026,
        REPORT_SERIES,
        REPORT_MONTH,
        "advance",
        PUBLISHER,
        None,
        None,
    ),
]

report_metadata_df = spark.createDataFrame(
    metadata_rows,
    schema=document_metadata_schema
)

display(report_metadata_df)

In [0]:
# ============================================================
# Attach actual document IDs from the processed corpus
# ============================================================

document_metadata_df = (
    document_ids_df
    .join(
        report_metadata_df,
        on="report_year",
        how="left"
    )
    .withColumn(
        "metadata_version",
        F.lit("v1")
    )
    .withColumn(
        "metadata_created_at",
        F.current_timestamp()
    )
    .select(
        "document_id",
        "report_series",
        "report_year",
        "report_month",
        "edition_status",
        "publisher",
        "publication_date",
        "data_cutoff_date",
        "metadata_version",
        "metadata_created_at",
    )
    .orderBy("report_year")
)

display(document_metadata_df)

In [0]:
# ============================================================
# Validate document metadata
# ============================================================

metadata_count = document_metadata_df.count()

if metadata_count != 5:
    raise RuntimeError(
        f"Expected 5 document metadata rows, "
        f"found {metadata_count}."
    )


# ------------------------------------------------------------
# Validate required fields.
# ------------------------------------------------------------

required_metadata_columns = [
    "document_id",
    "report_series",
    "report_year",
    "report_month",
    "edition_status",
    "publisher",
    "metadata_version",
]

for column_name in required_metadata_columns:

    null_count = (
        document_metadata_df
        .filter(
            F.col(column_name).isNull()
        )
        .count()
    )

    if null_count > 0:
        raise RuntimeError(
            f"Metadata column '{column_name}' "
            f"contains {null_count} null rows."
        )


# ------------------------------------------------------------
# Validate edition-status values.
# ------------------------------------------------------------

invalid_status_count = (
    document_metadata_df
    .filter(
        ~F.col("edition_status").isin(
            "final",
            "advance"
        )
    )
    .count()
)

if invalid_status_count > 0:
    raise RuntimeError(
        "Invalid edition_status values detected."
    )


# ------------------------------------------------------------
# Validate 2026 advance edition.
# ------------------------------------------------------------

status_2026 = (
    document_metadata_df
    .filter(
        F.col("report_year") == 2026
    )
    .select("edition_status")
    .first()["edition_status"]
)

if status_2026 != "advance":
    raise RuntimeError(
        "2026 GEP must be marked as "
        "edition_status='advance'."
    )


# ------------------------------------------------------------
# Validate 2022–2025 final editions.
# ------------------------------------------------------------

incorrect_final_count = (
    document_metadata_df
    .filter(
        (F.col("report_year").between(2022, 2025))
        &
        (F.col("edition_status") != "final")
    )
    .count()
)

if incorrect_final_count > 0:
    raise RuntimeError(
        "2022–2025 GEP reports must be "
        "edition_status='final'."
    )


print("Document metadata validation passed.")

In [0]:
# ============================================================
# Persist document metadata dimension
# ============================================================

(
    document_metadata_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DOCUMENT_METADATA_TABLE)
)

print(
    f"Saved document metadata: "
    f"{DOCUMENT_METADATA_TABLE}"
)

In [0]:
# ============================================================
# Enrich parent chunks with document metadata
# ============================================================

metadata_for_join_df = (
    document_metadata_df
    .select(
        "document_id",
        "report_series",
        "report_month",
        "edition_status",
        "publisher",
        "publication_date",
        "data_cutoff_date",
        "metadata_version",
    )
)

parent_enriched_df = (
    parent_df.alias("chunk")
    .join(
        metadata_for_join_df.alias("meta"),
        on="document_id",
        how="left"
    )
)

print(
    f"Enriched parent chunks: "
    f"{parent_enriched_df.count():,}"
)

display(
    parent_enriched_df
    .select(
        "parent_chunk_id",
        "document_id",
        "report_year",
        "report_series",
        "edition_status",
        "chapter",
        "region",
        "section",
        "subsection",
        "page_start",
        "page_end",
        "content_type",
    )
    .orderBy(
        "report_year",
        "page_start"
    )
    .limit(20)
)

In [0]:
# ============================================================
# Enrich child chunks with document metadata
# ============================================================

child_enriched_df = (
    child_df.alias("chunk")
    .join(
        metadata_for_join_df.alias("meta"),
        on="document_id",
        how="left"
    )
)

print(
    f"Enriched child chunks: "
    f"{child_enriched_df.count():,}"
)

display(
    child_enriched_df
    .select(
        "chunk_id",
        "parent_chunk_id",
        "document_id",
        "report_year",
        "report_series",
        "edition_status",
        "chapter",
        "region",
        "section",
        "subsection",
        "content_type",
        "page_start",
        "page_end",
    )
    .orderBy(
        "report_year",
        "page_start"
    )
    .limit(20)
)

In [0]:
# ============================================================
# Validate metadata enrichment
# ============================================================

parent_missing_metadata = (
    parent_enriched_df
    .filter(
        F.col("report_series").isNull()
        |
        F.col("edition_status").isNull()
    )
    .count()
)

child_missing_metadata = (
    child_enriched_df
    .filter(
        F.col("report_series").isNull()
        |
        F.col("edition_status").isNull()
    )
    .count()
)

print(
    "Parent chunks missing metadata:",
    parent_missing_metadata
)

print(
    "Child chunks missing metadata:",
    child_missing_metadata
)


if parent_missing_metadata != 0:
    raise RuntimeError(
        "Some parent chunks could not be mapped "
        "to document metadata."
    )

if child_missing_metadata != 0:
    raise RuntimeError(
        "Some child chunks could not be mapped "
        "to document metadata."
    )


# ------------------------------------------------------------
# Joining metadata must not change chunk counts.
# ------------------------------------------------------------

if parent_enriched_df.count() != parent_count:
    raise RuntimeError(
        "Parent chunk count changed during metadata join."
    )

if child_enriched_df.count() != child_count:
    raise RuntimeError(
        "Child chunk count changed during metadata join."
    )

print("Metadata enrichment validation passed.")

In [0]:
# ============================================================
# Validate parent-child relationships
# ============================================================

valid_parent_ids_df = (
    parent_enriched_df
    .select("parent_chunk_id")
    .distinct()
)

orphan_children_df = (
    child_enriched_df
    .select(
        "chunk_id",
        "parent_chunk_id"
    )
    .join(
        valid_parent_ids_df,
        on="parent_chunk_id",
        how="left_anti"
    )
)

orphan_count = orphan_children_df.count()

print(f"Orphan child chunks: {orphan_count:,}")

if orphan_count != 0:
    raise RuntimeError(
        f"Found {orphan_count} orphan child chunks."
    )

print("Parent-child relationship validation passed.")

In [0]:
# ============================================================
# Metadata distribution by report
# ============================================================

report_distribution_df = (
    child_enriched_df
    .groupBy(
        "report_year",
        "edition_status"
    )
    .agg(
        F.count("*").alias("child_chunks"),
        F.countDistinct(
            "parent_chunk_id"
        ).alias("parent_chunks_referenced"),
        F.countDistinct(
            "chapter"
        ).alias("chapters"),
        F.countDistinct(
            "region"
        ).alias("regions"),
    )
    .orderBy("report_year")
)

display(report_distribution_df)

In [0]:
# ============================================================
# Inspect region metadata
# ============================================================

region_distribution_df = (
    child_enriched_df
    .groupBy("region")
    .agg(
        F.count("*").alias("chunk_count")
    )
    .orderBy(
        F.desc("chunk_count")
    )
)

display(region_distribution_df)

In [0]:
# ============================================================
# Inspect section/subsection metadata
# ============================================================

section_distribution_df = (
    child_enriched_df
    .groupBy(
        "section",
        "subsection"
    )
    .agg(
        F.count("*").alias("chunk_count")
    )
    .orderBy(
        F.desc("chunk_count")
    )
)

display(
    section_distribution_df.limit(50)
)

In [0]:
# ============================================================
# Create retrieval_text
# ============================================================
#
# IMPORTANT:
#   chunk_text remains the source-faithful text.
#
#   retrieval_text is a derived representation that may be
#   evaluated later as an embedding input.
#
#   We are NOT generating embeddings in this notebook.
# ============================================================

child_enriched_df = (
    child_enriched_df
    .withColumn(
        "retrieval_text",
        F.concat_ws(
            "\n",
            
            F.concat(
                F.lit("Report: "),
                F.col("report_series"),
                F.lit(", "),
                F.col("report_year").cast("string")
            ),
            
            F.when(
                F.col("region").isNotNull(),
                F.concat(
                    F.lit("Region: "),
                    F.col("region")
                )
            ),
            
            F.when(
                F.col("chapter_title").isNotNull(),
                F.concat(
                    F.lit("Chapter: "),
                    F.col("chapter_title")
                )
            ),
            
            F.when(
                F.col("section").isNotNull(),
                F.concat(
                    F.lit("Section: "),
                    F.col("section")
                )
            ),
            
            F.when(
                F.col("subsection").isNotNull(),
                F.concat(
                    F.lit("Subsection: "),
                    F.col("subsection")
                )
            ),
            
            F.lit(""),
            
            F.col("chunk_text")
        )
    )
)

display(
    child_enriched_df
    .select(
        "chunk_id",
        "chunk_text",
        "retrieval_text"
    )
    .limit(5)
)

In [0]:
# ============================================================
# Validate source text fidelity
# ============================================================

source_text_df = (
    child_df
    .select(
        "chunk_id",
        F.col("chunk_text").alias(
            "original_chunk_text"
        )
    )
)

text_validation_df = (
    child_enriched_df
    .select(
        "chunk_id",
        "chunk_text"
    )
    .join(
        source_text_df,
        on="chunk_id",
        how="inner"
    )
)

modified_text_count = (
    text_validation_df
    .filter(
        F.col("chunk_text")
        !=
        F.col("original_chunk_text")
    )
    .count()
)

print(
    "Chunks whose source text changed:",
    modified_text_count
)

if modified_text_count != 0:
    raise RuntimeError(
        "Source chunk text was modified during "
        "metadata enrichment."
    )

print("Source-text fidelity validation passed.")

In [0]:
# ============================================================
# Add retrieval readiness fields
# ============================================================
#
# These fields make it easier to control what can later enter
# the RAG retrieval corpus.
#
# We preserve all chunks in Silver.
# We do NOT delete low-value chunks here.
# ============================================================

child_enriched_df = (
    child_enriched_df
    
    # Mark whether text exists.
    .withColumn(
        "has_text",
        F.length(
            F.trim(
                F.col("chunk_text")
            )
        ) > 0
    )
    
    # Character count for the retrieval representation.
    .withColumn(
        "retrieval_character_count",
        F.length(
            F.col("retrieval_text")
        )
    )
    
    # Approximate token count.
    # ~4 characters/token is only a rough operational estimate.
    .withColumn(
        "approx_retrieval_token_count",
        F.ceil(
            F.length(
                F.col("retrieval_text")
            ) / F.lit(4.0)
        ).cast("integer")
    )
    
    # Mark obvious front matter.
    # We preserve it rather than deleting it.
    .withColumn(
        "is_front_matter",
        F.col("content_type") == "front_matter"
    )
    
    # Baseline retrieval eligibility.
    #
    # This is intentionally conservative.
    # Later experiments can change retrieval filtering.
    .withColumn(
        "baseline_retrieval_eligible",
        (
            F.length(
                F.trim(
                    F.col("chunk_text")
                )
            ) > 0
        )
        &
        (
            F.col("content_type")
            != "front_matter"
        )
    )
)

In [0]:
# ============================================================
# Retrieval readiness summary
# ============================================================

retrieval_summary_df = (
    child_enriched_df
    .agg(
        F.count("*").alias(
            "total_chunks"
        ),
        
        F.sum(
            F.when(
                F.col("baseline_retrieval_eligible"),
                1
            ).otherwise(0)
        ).alias(
            "baseline_eligible"
        ),
        
        F.sum(
            F.when(
                F.col("is_front_matter"),
                1
            ).otherwise(0)
        ).alias(
            "front_matter_chunks"
        ),
        
        F.sum(
            F.when(
                F.length(
                    F.col("chunk_text")
                ) < 300,
                1
            ).otherwise(0)
        ).alias(
            "chunks_under_300_chars"
        ),
        
        F.avg(
            "approx_retrieval_token_count"
        ).alias(
            "avg_retrieval_tokens"
        ),
    )
)

display(retrieval_summary_df)

In [0]:
# ============================================================
# Persist enriched parent chunks
# ============================================================

(
    parent_enriched_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PARENT_ENRICHED_TABLE)
)

print(
    f"Saved enriched parent chunks: "
    f"{PARENT_ENRICHED_TABLE}"
)

In [0]:
# ============================================================
# Persist enriched child chunks
# ============================================================

(
    child_enriched_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CHILD_ENRICHED_TABLE)
)

print(
    f"Saved enriched child chunks: "
    f"{CHILD_ENRICHED_TABLE}"
)

In [0]:
# ============================================================
# Read persisted tables back from Unity Catalog
# ============================================================

saved_metadata_df = spark.table(
    DOCUMENT_METADATA_TABLE
)

saved_parent_df = spark.table(
    PARENT_ENRICHED_TABLE
)

saved_child_df = spark.table(
    CHILD_ENRICHED_TABLE
)


saved_metadata_count = (
    saved_metadata_df.count()
)

saved_parent_count = (
    saved_parent_df.count()
)

saved_child_count = (
    saved_child_df.count()
)


print(
    f"Saved document metadata: "
    f"{saved_metadata_count:,}"
)

print(
    f"Saved parent chunks: "
    f"{saved_parent_count:,}"
)

print(
    f"Saved child chunks: "
    f"{saved_child_count:,}"
)


# ------------------------------------------------------------
# Validate persisted counts.
# ------------------------------------------------------------

assert saved_metadata_count == 5

assert saved_parent_count == parent_count

assert saved_child_count == child_count

print("Persisted-table validation passed.")

In [0]:
# ============================================================
# Final metadata quality check
# ============================================================

final_quality_df = (
    saved_child_df
    .groupBy(
        "report_year",
        "edition_status"
    )
    .agg(
        F.count("*").alias(
            "chunks"
        ),
        
        F.countDistinct(
            "document_id"
        ).alias(
            "documents"
        ),
        
        F.countDistinct(
            "parent_chunk_id"
        ).alias(
            "parents"
        ),
        
        F.sum(
            F.when(
                F.col("region").isNotNull(),
                1
            ).otherwise(0)
        ).alias(
            "chunks_with_region"
        ),
        
        F.sum(
            F.when(
                F.col("section").isNotNull(),
                1
            ).otherwise(0)
        ).alias(
            "chunks_with_section"
        ),
        
        F.sum(
            F.when(
                F.col(
                    "baseline_retrieval_eligible"
                ),
                1
            ).otherwise(0)
        ).alias(
            "retrieval_eligible"
        ),
    )
    .orderBy(
        "report_year"
    )
)

display(final_quality_df)

In [0]:
# ============================================================
# 04_document_metadata completion summary
# ============================================================

print("=" * 72)
print("04_document_metadata COMPLETED")
print("=" * 72)

print(
    f"Documents:        "
    f"{saved_metadata_count:,}"
)

print(
    f"Parent chunks:    "
    f"{saved_parent_count:,}"
)

print(
    f"Child chunks:     "
    f"{saved_child_count:,}"
)

print()

print(
    f"Metadata table:   "
    f"{DOCUMENT_METADATA_TABLE}"
)

print(
    f"Parent table:     "
    f"{PARENT_ENRICHED_TABLE}"
)

print(
    f"Child table:      "
    f"{CHILD_ENRICHED_TABLE}"
)

print("=" * 72)

print(
    "Silver unstructured processing is complete."
)

print(
    "Next phase: 04_rag/00_mlflow_experiment_setup"
)

print("=" * 72)